#Initialization

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [2]:
import os
os.environ["PYTHONHASHSEED"] = "123"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [3]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [4]:
print("Device:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("PyTorch version:", torch.__version__)

Device: Tesla T4
CUDA version: 12.8
cuDNN version: 91900
PyTorch version: 2.11.0+cu128


In [5]:
sc3_result = []

#Data Loading

##Sc.3. Data

In [6]:
val_sc3_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc3/val_sc3.xlsx")
test_sc3_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc3/test_sc3.xlsx")


#Pre-processing Set

In [7]:
SEED = 123
EPOCHS = 20

In [8]:
# Transformation

IMAGE_SIZE = 224

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

In [9]:
# Dataset Loader

BATCH_SIZE = 32
NUM_WORKERS = 2

class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        image  = Image.open(sample["path"]).convert("RGB")
        image  = self.transform(image) if self.transform else transforms.ToTensor()(image)
        label  = int(sample["label"])
        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "child_id": str(sample["child_id"]),
            "path": str(sample["path"]),
        }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataframe, transform, shuffle):
    ds = FaceDataset(dataframe, transform=transform)
    g  = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g,
        worker_init_fn=seed_worker,  # ← ADD THIS
    )

#Model Testing

In [10]:
# Prediction: Function

def collect_predictions(model, loader):
    model.eval()
    rows = []
    with torch.no_grad():
        for batch in loader:
            xb = batch["image"].to(device, non_blocking=True)
            yb = batch["label"].cpu().numpy().astype(int)
            logits = model(xb)
            probs  = torch.softmax(logits, dim=1).detach().cpu().numpy()
            logits_np = logits.detach().cpu().numpy()

            for i in range(len(yb)):
                rows.append({
                    "child_id": batch["child_id"][i],
                    "path": batch["path"][i],
                    "label": int(yb[i]),
                    "logit0": float(logits_np[i, 0]),
                    "logit1": float(logits_np[i, 1]),
                    "prob0": float(probs[i, 0]),
                    "prob1": float(probs[i, 1])
                })
    return pd.DataFrame(rows)

In [11]:
#Evaluation Function

def safe_div(num, den):
    return float(num) / float(den) if den else 0.0

def metric_bundle(y_true, prob1, threshold):
    y_true = np.asarray(y_true).astype(int)
    prob1 = np.asarray(prob1).astype(float)
    y_pred = (prob1 >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy = safe_div(tn + tp, len(y_true))
    precision = safe_div(tp, tp + fp)
    sensitivity = safe_div(tp, tp + fn)   # recall for stunting
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * sensitivity, precision + sensitivity)
    beta2 = 2.0
    f2 = safe_div((1 + beta2**2) * precision * sensitivity, (beta2**2) * precision + sensitivity)
    bal_acc = 0.5 * (sensitivity + specificity)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,
        "balanced_accuracy": bal_acc,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


In [12]:
# Model - pre-trained EfficientNet B0

def create_model(num_classes=2):
    weights = EfficientNet_B0_Weights.DEFAULT
    model   = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

#Sc.3. EfficientNet-B0

##Sc.3. EfficientNet-B0 with SGD

In [13]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 149MB/s]


In [14]:
# Load Dataset
dl_val_sc3   = make_loader(val_sc3_df, eval_tfms, shuffle=False)
dl_test_sc3  = make_loader(test_sc3_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc3_df)} | test={len(test_sc3_df)}")


val=10 | test=10


In [15]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc3_sgd.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc3)
test_predict  = collect_predictions(model, dl_test_sc3)

###Sc.3. EfficientNet-B0 with SGD Validation Data

In [16]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.6,
 'precision': 0.5714285714285714,
 'sensitivity': 0.8,
 'specificity': 0.4,
 'f1': 0.6666666666666666,
 'f2': 0.7407407407407408,
 'balanced_accuracy': 0.6000000000000001,
 'TN': 2,
 'FP': 3,
 'FN': 1,
 'TP': 4}

In [17]:
sc3_result.append([
    "sc3-efficientnet-b0-sgd",
    "sc3", "efficientnet-b0", "sgd",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.3. EfficientNet-B0 with SGD Test Data

In [18]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.6,
 'precision': 0.5714285714285714,
 'sensitivity': 0.8,
 'specificity': 0.4,
 'f1': 0.6666666666666666,
 'f2': 0.7407407407407408,
 'balanced_accuracy': 0.6000000000000001,
 'TN': 2,
 'FP': 3,
 'FN': 1,
 'TP': 4}

In [19]:
sc3_result.append([
    "sc3-efficientnet-b0-sgd",
    "sc3", "efficientnet-b0", "sgd",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

##Sc.3. EfficientNet-B0 with Adam

In [20]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [21]:
# Load Dataset
dl_val_sc3   = make_loader(val_sc3_df, eval_tfms, shuffle=False)
dl_test_sc3  = make_loader(test_sc3_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc3_df)} | test={len(test_sc3_df)}")


val=10 | test=10


In [22]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc3_adam.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc3)
test_predict  = collect_predictions(model, dl_test_sc3)

###Sc.3. EfficientNet-B0 with Adam Validation data

In [23]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.6,
 'precision': 0.5555555555555556,
 'sensitivity': 1.0,
 'specificity': 0.2,
 'f1': 0.7142857142857143,
 'f2': 0.8620689655172413,
 'balanced_accuracy': 0.6,
 'TN': 1,
 'FP': 4,
 'FN': 0,
 'TP': 5}

In [24]:
sc3_result.append([
    "sc3-efficientnet-b0-adam",
    "sc3", "efficientnet-b0", "adam",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.3. EfficientNet-B0 with Adam Test Data

In [25]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.6,
 'precision': 0.5555555555555556,
 'sensitivity': 1.0,
 'specificity': 0.2,
 'f1': 0.7142857142857143,
 'f2': 0.8620689655172413,
 'balanced_accuracy': 0.6,
 'TN': 1,
 'FP': 4,
 'FN': 0,
 'TP': 5}

In [26]:
sc3_result.append([
    "sc3-efficientnet-b0-adam",
    "sc3", "efficientnet-b0", "adam",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

##Sc.3. EfficientNet-B0 with AdamW

In [27]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


In [28]:
# Load Dataset
dl_val_sc3   = make_loader(val_sc3_df, eval_tfms, shuffle=False)
dl_test_sc3  = make_loader(test_sc3_df, eval_tfms, shuffle=False)

print(f"\nval={len(val_sc3_df)} | test={len(test_sc3_df)}")


val=10 | test=10


In [29]:
# Prediction on validation and test data
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc3_adamw.pth"

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

val_predict   = collect_predictions(model, dl_val_sc3)
test_predict  = collect_predictions(model, dl_test_sc3)

###Sc.3. EfficientNet-B0 with AdamW Validation Data

In [30]:
# Evaluation for Validation data
print('validation evaluation:')
result = metric_bundle(val_predict.label, val_predict.prob1, threshold=0.5)
result

validation evaluation:


{'accuracy': 0.6,
 'precision': 0.5555555555555556,
 'sensitivity': 1.0,
 'specificity': 0.2,
 'f1': 0.7142857142857143,
 'f2': 0.8620689655172413,
 'balanced_accuracy': 0.6,
 'TN': 1,
 'FP': 4,
 'FN': 0,
 'TP': 5}

In [31]:
sc3_result.append([
    "sc3-efficientnet-b0-adamw",
    "sc3", "efficientnet-b0", "adamw",
    "validation",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

###Sc.3. EfficientNet-B0 with AdamW Test Data

In [32]:
# Evaluation for Test data
print('test evaluation:')
result = metric_bundle(test_predict.label, test_predict.prob1, threshold=0.5)
result

test evaluation:


{'accuracy': 0.4,
 'precision': 0.42857142857142855,
 'sensitivity': 0.6,
 'specificity': 0.2,
 'f1': 0.5,
 'f2': 0.5555555555555556,
 'balanced_accuracy': 0.4,
 'TN': 1,
 'FP': 4,
 'FN': 2,
 'TP': 3}

In [33]:
sc3_result.append([
    "sc3-efficientnet-b0-adamw",
    "sc3", "efficientnet-b0", "adamw",
    "test",
    result["accuracy"],
    result["precision"],
    result["sensitivity"],
    result["specificity"],
    result["f1"],
    result["f2"],
    result["TN"],
    result["FP"],
    result["FN"],
    result["TP"],
])

#Result

In [34]:
sc3_result_df = pd.DataFrame(sc3_result, columns=['model_name',
                                                  'scheme', 'model', 'optimizer',
                                                  'evaluation',
                                                  'accuracy',
                                                  'precision',
                                                  'sensitivity',
                                                  'specificity',
                                                  'f1',
                                                  'f2',
                                                  'TN',
                                                  'FP',
                                                  'FN',
                                                  'TP',
                                                  ])
display(sc3_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc3_result_df.to_excel(base + 'sc3_result.xlsx', index=False)

,model_name,scheme,model,optimizer,evaluation,accuracy,precision,sensitivity,specificity,f1,f2,TN,FP,FN,TP
0,sc3-efficientnet-b0-sgd,sc3,efficientnet-b0,sgd,validation,0.6,0.571429,0.8,0.4,0.666667,0.740741,2,3,1,4
1,sc3-efficientnet-b0-sgd,sc3,efficientnet-b0,sgd,test,0.6,0.571429,0.8,0.4,0.666667,0.740741,2,3,1,4
2,sc3-efficientnet-b0-adam,sc3,efficientnet-b0,adam,validation,0.6,0.555556,1.0,0.2,0.714286,0.862069,1,4,0,5
3,sc3-efficientnet-b0-adam,sc3,efficientnet-b0,adam,test,0.6,0.555556,1.0,0.2,0.714286,0.862069,1,4,0,5
4,sc3-efficientnet-b0-adamw,sc3,efficientnet-b0,adamw,validation,0.6,0.555556,1.0,0.2,0.714286,0.862069,1,4,0,5
5,sc3-efficientnet-b0-adamw,sc3,efficientnet-b0,adamw,test,0.4,0.428571,0.6,0.2,0.500000,0.555556,1,4,2,3
